# UC Myeloid: CytoTRACE2 Potency Scoring and Palantir Pseudotime (Fig. 1G–H)

Two complementary analyses on CD68+ myeloid cells from the UC atlas:

1. **Palantir pseudotime** (Fig. 1H): diffusion-based trajectory from a
   Mo-Mac / Mac S+XS- root, capturing the monocyte-to-macrophage continuum.
2. **CytoTRACE2 potency scoring** (Fig. 1G): cell-potency / differentiation
   state estimated from raw transcript counts.

Pericytes, neutrophils, cycling myeloid cells, and rare DCs are excluded to
focus on the core macrophage trajectory.

**Inputs** (exported from `02_myeloid_subclustering.R`):
- `umap_myeloid_iter3_anno.csv`  – UMAP coordinates
- `mye_harmony_iter3.csv`        – Harmony embedding (30 dims)
- `uc_atlas_annotated.h5ad`      – full annotated atlas (for raw counts)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import palantir
from matplotlib.colors import LinearSegmentedColormap
from cytotrace2_py.cytotrace2_py import cytotrace2
import fast_matrix_market as fmm
import scipy.sparse

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/uc/scrna/output"
MYE_DIR    = f"{DATA_DIR}/myeloid"
OUTPUT_DIR = f"{DATA_DIR}/myeloid"

## 1. Load myeloid cells and attach embeddings

In [ ]:
# Load full atlas and subset to myeloid
scrna = sc.read_h5ad(f"{DATA_DIR}/uc_atlas_annotated.h5ad")
mye   = scrna[scrna.obs["cell_category"] == "Myeloid"].copy()

print(mye.obs["cell_type_short"].value_counts())

In [ ]:
# Attach myeloid UMAP coordinates (exported from R subclustering)
umap_mye = pd.read_csv(f"{MYE_DIR}/umap_myeloid_iter3_anno.csv")
mye.obsm["X_umap"] = umap_mye[["umapharmonymyeiter3_1",
                                  "umapharmonymyeiter3_2"]].to_numpy()

# Attach Harmony embedding
harm = pd.read_csv(f"{MYE_DIR}/mye_harmony_iter3.csv", index_col=0)
mye.obsm["X_harmony"] = harm.to_numpy()

In [ ]:
# Exclude cycling, neutrophils, and rare DC populations for trajectory analysis
exclude = ["Cycl Myeloid", "Neutrophil", "cDC1", "CCR7+ Mat DC"]
mye = mye[~mye.obs["cell_type_short"].isin(exclude)].copy()

print(mye.obs["cell_type_short"].value_counts())

In [ ]:
# Quick UMAP check
sc.pl.umap(mye, color="cell_type_short", frameon=False, s=5)

## 2. Palantir pseudotime (Fig. 1H)

Root cell: centroid of Mac S+XS- cluster (most stem-like state by CytoTRACE2).

In [ ]:
# Diffusion maps and multi-scale space
dm_res   = palantir.utils.run_diffusion_maps(mye, n_components=5, pca_key="X_harmony")
ms_data  = palantir.utils.determine_multiscale_space(mye)
imputed_X = palantir.utils.run_magic_imputation(mye)

In [ ]:
# Find cell closest to Mac S+XS- UMAP centroid as root
coords = mye.obsm["X_umap"]
early_mask = mye.obs["cell_type_short"] == "Mac S+XS-"
early_idx  = np.where(early_mask)[0]
centroid   = coords[early_idx].mean(axis=0)
root_cell  = mye.obs_names[early_idx[np.argmin(
    ((coords[early_idx] - centroid) ** 2).sum(axis=1)
)]]
print(f"Root cell: {root_cell}")

In [ ]:
# Highlight root on UMAP
start_states = pd.Series(["Mac S+XS- (root)"], index=[root_cell])
palantir.plot.highlight_cells_on_umap(mye, start_states, s=5)
plt.show()

In [ ]:
# Run Palantir
pr_res = palantir.core.run_palantir(mye, root_cell, num_waypoints=500)

# Plot pseudotime + branch probabilities
palantir.plot.plot_palantir_results(mye, s=3)
plt.show()

## 3. DPT validation (diffusion pseudotime with PAGA)

Validates the Palantir trajectory using scanpy's diffusion pseudotime (DPT)
with PAGA-initialized UMAP.

In [ ]:
mye_cp = mye.copy()

# Build neighbor graph and PAGA on Harmony embedding
sc.pp.neighbors(mye_cp, use_rep="X_harmony", n_neighbors=20, n_pcs=25)
sc.tl.paga(mye_cp, groups="cell_type_short")
sc.pl.paga(mye_cp, plot=False)   # compute positions without plotting
sc.tl.umap(mye_cp, init_pos="paga")

# DPT from Mac S+XS- root
mye_cp.uns["iroot"] = np.flatnonzero(
    mye_cp.obs["cell_type_short"] == "Mac S+XS-"
)[0]
sc.tl.dpt(mye_cp)

# Restore original UMAP for consistent visualization
mye_cp.obsm["X_umap"] = mye.obsm["X_umap"]

In [ ]:
umap_purple = LinearSegmentedColormap.from_list(
    "umap_purple", ["#e8d4cf", "#a16f8d", "#2d223a"]
)
ax = sc.pl.umap(
    mye_cp, color="dpt_pseudotime",
    cmap=umap_purple, vmin=0, vmax="p99",
    na_color="#e8d4cf", frameon=False, show=False
)
ax.set_title("Diffusion Pseudotime (DPT validation)", fontsize=14)
plt.show()

## 4. CytoTRACE2 potency scoring (Fig. 1G)

Estimates developmental potency from raw transcript counts.
Higher score = more stem-like / less differentiated.

In [ ]:
# Prepare clean barcodes and export for CytoTRACE2
mye_clean = mye.copy()
mye_clean.obs_names = [f"cell_{i}" for i in range(mye_clean.n_obs)]

# Annotation file (cell_id → cell type)
annot_df = pd.DataFrame(
    {"celltype": mye_clean.obs["cell_type_short"].values},
    index=mye_clean.obs_names
)
annot_df.to_csv(f"{OUTPUT_DIR}/mac_annotations.txt", sep="\t")

# Raw counts matrix (genes × cells) via MTX
raw_X = scipy.sparse.csc_matrix(mye_clean.raw.X).T   # genes × cells
fmm.mmwrite(f"{OUTPUT_DIR}/mye_raw_counts.mtx", raw_X)
pd.Series(mye_clean.var_names.tolist()).to_csv(
    f"{OUTPUT_DIR}/mye_raw_counts_genes.txt", index=False, header=False
)
pd.Series(mye_clean.obs_names.tolist()).to_csv(
    f"{OUTPUT_DIR}/mye_raw_counts_barcodes.txt", index=False, header=False
)

# Reload as dense DataFrame (required by cytotrace2)
counts  = fmm.mmread(f"{OUTPUT_DIR}/mye_raw_counts.mtx")
genes   = pd.read_csv(f"{OUTPUT_DIR}/mye_raw_counts_genes.txt",    header=None)[0].tolist()
barcodes = pd.read_csv(f"{OUTPUT_DIR}/mye_raw_counts_barcodes.txt", header=None)[0].tolist()
expr_df = pd.DataFrame(counts.toarray(), index=genes, columns=barcodes)
expr_df.to_csv(f"{OUTPUT_DIR}/mac_expression.txt", sep="\t")

In [ ]:
# Run CytoTRACE2
results = cytotrace2(
    f"{OUTPUT_DIR}/mac_expression.txt",
    annotation_path=f"{OUTPUT_DIR}/mac_annotations.txt",
    species="human",
)

# Attach scores back to mye object
mye.obs["cytotrace2_score"]          = results["CytoTRACE2_Score"].tolist()
mye.obs["cytotrace2_score_relative"] = results["CytoTRACE2_Relative"].tolist()

# Per-cluster means
print(mye.obs.groupby("cell_type_short")["cytotrace2_score"]
      .mean().sort_values(ascending=False))

results.to_csv(f"{OUTPUT_DIR}/mac_cytotrace2_scores.csv")

In [ ]:
# Order cell types by mean CytoTRACE2 score (most potent first)
ct_order = (
    mye.obs.groupby("cell_type_short")["cytotrace2_score"]
    .mean().sort_values(ascending=False).index.tolist()
)

fig, ax = plt.subplots(figsize=(6.2, 4))
sns.boxplot(
    data=mye.obs, x="cell_type_short", y="cytotrace2_score",
    order=ct_order, palette="Blues_r",
    width=0.5, showcaps=False,
    boxprops=dict(alpha=0.6), whiskerprops=dict(alpha=0.6),
    flierprops=dict(marker=""), ax=ax
)
sns.stripplot(
    data=mye.obs, x="cell_type_short", y="cytotrace2_score",
    order=ct_order, color="steelblue",
    size=1.5, jitter=True, alpha=0.3, ax=ax, zorder=3
)
ax.set_xlabel("Cell Type", fontsize=11)
ax.set_ylabel("CytoTRACE2 Plasticity Score", fontsize=12)
ax.set_xticklabels(ct_order, rotation=35, ha="right", fontsize=11)
ax.grid(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig1g_cytotrace2_myeloid_boxplot.pdf", bbox_inches="tight")
plt.show()